In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pyrepseq as prs
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages

In [2]:
# Setting paths
data_dir = Path("../Data/20260116 Comparison 3")
output_dir = Path("Comparison3_Figures")
output_dir.mkdir(exist_ok=True)

In [3]:
markers = [
        "RiO-Allo:H-2Kb-ATLVFHNL-pAbO",
        "RiO-Allo:H-2Kb-EEEPVKKI-pAbO",
        "RiO-Allo:H-2Kb-HIYEFPQL-pAbO",
        "RiO-Allo:H-2Kb-INFDFPKL-pAbO",
        "RiO-Allo:H-2Kb-RAYLFNSV-pAbO",
        "RiO-Allo:H-2Kb-RTYTYEKL-pAbO",
    	"RiO-Allo:H-2Kb-SNYLFTKL-pAbO",
        "RiO-Allo:H-2Kb-SSYTFPKM-pAbO",
        "RiO-Allo:H-2Kb-SVYVYKVL-pAbO",
        "RiO-Allo:H-2Kb-VAFDFTKV-pAbO",
        "RiO-Allo:H-2Kb-VGPRYTNL-pAbO",
        "RiO-Allo:H-2Kb-VIVRFLTV-pAbO",
        "RiO-Allo:H-2Kb-VSFTYRYL-pAbO"
        ]

peptides = [
    "ATLVFHNL",
    "EEEPVKKI",
    "HIYEFPQL",
    "INFDFPKL",
    "RAYLFNSV",
    "RTYTYEKL",
    "SNYLFTKL",
    "SSYTFPKM",
    "SVYVYKVL",
    "VAFDFTKV",
    "VGPRYTNL",
    "VIVRFLTV",
    "VSFTYRYL"
]

In [4]:
df = pd.read_csv(data_dir / "20251223 BL6-B10BR HTx HIL Clonotypes with ADT Counts.csv")

df = df.sort_values(by=["CTCount","Ly-49C-Ly-49I-Klra3-Klra9-AMM2139-pAbO"], ascending=[False, False])

In [5]:
counts = df['TCRClonotype'].value_counts()
filtered = counts[counts>=5].index.tolist()
filtered_df = df[df['TCRClonotype'].isin(filtered)]

In [6]:
summary_dicts = []

for c in filtered:

    subdf = filtered_df[filtered_df['TCRClonotype'] == c]

    for p in markers:

        values = subdf[p]

        mean = np.mean(values)
        n = int(counts.loc[c])
        sem = np.std(values,ddof=1) / np.sqrt(n)

        summary = {
            "TCRClonotype": c,
            "peptide": p,
            "mean": mean,
            "sem": sem,
            "n": n
            }
        
        summary_dicts.append(summary)

summary_df = pd.DataFrame(summary_dicts)

In [7]:
groups = {k: g for k, g in summary_df.groupby('TCRClonotype', sort=False)}

xs = np.arange(len(peptides))
pdf_name = 'clonotypes_5_counts.pdf'

with PdfPages(pdf_name) as pdf:
    for start in range(0,len(filtered), 32):
        chunk = filtered[start:start + 32]

        fig, axes = plt.subplots(8, 4, figsize=(18,30))
        axes = axes.flatten()

        for ax_i, c in enumerate(chunk):
            ax = axes[ax_i]
            plotdf = groups[c].set_index("peptide").reindex(markers)

            means = plotdf["mean"].to_numpy()
            errors = plotdf["sem"].to_numpy()

            ax.bar(xs, means)
            ax.errorbar(xs, means, yerr=errors, fmt="none", capsize=2, linewidth=0.8)

            n_occ = int(counts.loc[c])
            ax.set_title(f"{c}\n(n={n_occ})", fontsize=7)

            ax.set_xticks(xs)
            ax.set_xticklabels(peptides, rotation=90, fontsize=6)
            ax.tick_params(axis="y", labelsize=6)
            ax.spines["top"].set_visible(False)
            ax.spines["right"].set_visible(False)

        for j in range(len(chunk), 32):
            axes[j].axis("off")

        fig.tight_layout()
        pdf.savefig(fig)
        plt.close(fig)
